In [1]:
#import necessary libraries
from pytesseract import Output
import pytesseract
from PIL import Image
import os
import pandas as pd
import cv2
import Levenshtein

In [2]:
df = pd.read_csv("MOCKAROO_DATA.csv", encoding='cp1252')
last_names = df['last_name'].tolist()
print(last_names)

['Mousby', 'Mapledoram', 'Gerty', 'Wyllie', 'Blakeney', 'Pywell', 'Lehenmann', 'Itzcovich', 'Alenov', 'Ivanonko', 'Trehearne', 'Dolley', 'Stockey', 'Linfitt', 'Brooksbie', 'Henriquet', 'Landeg', 'Cunio', 'Medgewick', 'Skipp', 'Walcar', 'Melloy', 'Durnford', 'Norcop', 'Cominello', 'Kupper', 'Ragborne', 'Glassman', 'Saddleton', 'Seiller', 'Sarre', 'Simion', 'Ratchford', 'Klimkovich', 'Weddell', 'Arnald', 'Bobasch', 'Spours', 'Sanbrooke', 'Mackstead', 'Klimpke', 'Meaders', 'Blumfield', 'Clackson', 'Garey', 'Pavyer', 'Oakes', 'Romaines', "O'Spellissey", 'Leathe', 'Bullivant', 'Akess', 'Taphouse', 'Hukins', 'Prettejohns', 'Haresnape', 'Germain', 'Noden', 'MacCartan', 'Dellow', 'Darwen', 'Keneforde', 'Sneden', 'Choules', 'Renison', 'Pottes', 'Tebbut', 'Hush', 'Jagson', 'Lasham', 'Lobb', 'Huws', 'Bernardeschi', 'Klos', 'Begent', 'Walas', 'Bosket', 'Brannon', 'Gabby', 'Giacobbo', 'Rollitt', 'Revely', 'Goford', 'Gosswell', 'Luggar', 'Crunkhorn', 'Sorensen', 'Redihough', 'Carragher', 'Mattiazzi'

In [3]:
#check if "Last Name" is found. If match, compute error rate; otherwise, just note na hindi kita. Na-affect ng noise
def compute_error(text, csv_idx):
    match = 0
    if "Last Name" in text:
        match = 1
        #split the text into words
        words = text.split()
        
        start_idx = words.index("Name")

        #get length of last name from csv
        csv_last_name = last_names[csv_idx]
        csv_last_name_length = len(csv_last_name)
        num_words = len((csv_last_name.split(" ")))
        #extract last name form text
        found_last_name = " ".join(words[start_idx + 1: start_idx + 1 + num_words])
        

        #compute character error rate 
        char_diff = Levenshtein.distance(found_last_name, csv_last_name)
        cer = char_diff/csv_last_name_length
        
        #compute word error rate
        word_diff = Levenshtein.distance(found_last_name.split(), csv_last_name.split())
        word_count = len(csv_last_name.split())
        wer = word_diff/word_count
        return [cer, wer, match]
    return [0, 0, match]

<h2> Brightness w/o Pre-processing</h2>

In [4]:
# Specify the path to the Tesseract executable
pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/opt/tesseract/bin/tesseract'


In [5]:

#run OCR on brightness case

#load image
folder = 'Mail Merged Forms with Effects/'
bright_pattern = 'form page 1-bright-{0:03d}.jpg'

#init var 
total_cer = 0 
total_wer = 0
num_images = 200
num_matches = 0

for i in range (1, num_images+1):
    filename = bright_pattern.format(i)
    image_path = os.path.join(folder, filename)

    if not os.path.exists(image_path):  # Check if file exists
        print(f"Skipping missing file: {filename}")
        continue

    #Load image 
    image = Image.open(image_path)

    #set bounding box
    bbox_form = (0, 370, image.size[0], 1140) # Bounding Box for 02021e
    bbox_image = image.crop(bbox_form)

    #run OCR and compute error
    text = pytesseract.image_to_string(bbox_image)
    err = compute_error(text, i-1)
    cer = err[0]
    wer = err[1]
    num_matches += err[2]
    num_misses = 200-num_matches
    
    total_cer += cer
    total_wer += wer
    

print(f"Total CER: {total_cer}")
average_cer = total_cer/num_images
print(f"Average CER: {average_cer}")
print(f"Total WER: {total_wer}")
average_wer = total_wer/num_images
print(f"Average WER: {average_wer}")
print(f"Total Matches: {num_matches}")
print(f"Total Misses: {num_misses}")

Total CER: 0.4583333333333333
Average CER: 0.0022916666666666667
Total WER: 2.0
Average WER: 0.01
Total Matches: 190
Total Misses: 10


<h2> Tint w/o Pre-processing</h2>

In [8]:
#load image
folder = 'Mail Merged Forms with Effects/'
bright_pattern = 'form page 1-yellow-{0:03d}.jpg'

#init var 
total_cer = 0 
total_wer = 0
num_images = 200
num_matches = 0

for i in range (1, num_images+1):
    filename = bright_pattern.format(i)
    image_path = os.path.join(folder, filename)

    if not os.path.exists(image_path):  # Check if file exists
        print(f"Skipping missing file: {filename}")
        continue

    #Load image 
    image = Image.open(image_path)

    #set bounding box
    bbox_form = (0, 370, image.size[0], 1140) # Bounding Box for 02021e
    bbox_image = image.crop(bbox_form)

    #run OCR and compute error
    text = pytesseract.image_to_string(bbox_image)
    err = compute_error(text, i-1)
    cer = err[0]
    wer = err[1]
    num_matches += err[2]
    num_misses = 200-num_matches
    
    total_cer += cer
    total_wer += wer
    

print(f"Total CER: {total_cer}")
average_cer = total_cer/num_images
print(f"Average CER: {average_cer}")
print(f"Total WER: {total_wer}")
average_wer = total_wer/num_images
print(f"Average WER: {average_wer}")
print(f"Total Matches: {num_matches}")
print(f"Total Misses: {num_misses}")

Total CER: 0.39166666666666666
Average CER: 0.001958333333333333
Total WER: 3.0
Average WER: 0.015
Total Matches: 200
Total Misses: 0


<h2>Rotation w/o Pre-processing</h2>

In [12]:
#load image
folder = 'Mail Merged Forms with Effects/'
bright_pattern = 'form page 1-rotate-{0:03d}.jpg'

#init var 
total_cer = 0 
total_wer = 0
num_images = 200
num_matches = 0

for i in range (1, num_images+1):
    filename = bright_pattern.format(i)
    image_path = os.path.join(folder, filename)

    if not os.path.exists(image_path):  # Check if file exists
        print(f"Skipping missing file: {filename}")
        continue

    #Load image 
    image = Image.open(image_path)

    #set bounding box
    bbox_form = (0, 370, image.size[0], 1140) # Bounding Box for 02021e
    bbox_image = image.crop(bbox_form)

    #run OCR and compute error
    text = pytesseract.image_to_string(bbox_image)
    err = compute_error(text, i-1)
    cer = err[0]
    wer = err[1]
    matches = err[2]
    if (matches == 1):
        num_matches += 1
    miss = 200-num_matches
  
    total_cer += cer
    total_wer += wer

print(f"Total CER: {total_cer}")
average_cer = total_cer/num_images
print(f"Average CER: {average_cer}")
print(f"Total WER: {total_wer}")
average_wer = total_wer/num_images
print(f"Average WER: {average_wer}")
print(f"Total Matches: {num_matches}")
print(f"Total Misses: {miss}")

Total CER: 4.041666666666666
Average CER: 0.02020833333333333
Total WER: 4.0
Average WER: 0.02
Total Matches: 14
Total Misses: 186


<h2> Distortion w/o Pre-Processing </h2>

In [13]:
#load image
folder = 'Mail Merged Forms with Effects/'
bright_pattern = 'form page 1-noise-{0:03d}.jpg'

#init var 
total_cer = 0 
total_wer = 0
num_images = 200
num_matches = 0
is_match = 0
for i in range (1, num_images+1):
    filename = bright_pattern.format(i)
    image_path = os.path.join(folder, filename)

    if not os.path.exists(image_path):  # Check if file exists
        print(f"Skipping missing file: {filename}")
        continue

    #Load image 
    image = Image.open(image_path)
    err = compute_error(text, i-1)

    #while err[2] == 0:
        #rotate by 2 deg
        #OCR on image
        
    #set bounding box
    bbox_form = (0, 370, image.size[0], 1140) # Bounding Box for 02021e
    bbox_image = image.crop(bbox_form)

    #run OCR and compute error
    text = pytesseract.image_to_string(bbox_image)
    err = compute_error(text, i-1)
    cer = err[0]
    wer = err[1]
    num_matches += err[2]
    num_misses = 200-num_matches
    
    total_cer += cer
    total_wer += wer
    

print(f"Total CER: {total_cer}")
average_cer = total_cer/num_images
print(f"Average CER: {average_cer}")
print(f"Total WER: {total_wer}")
average_wer = total_wer/num_images
print(f"Average WER: {average_wer}")
print(f"Total Matches: {num_matches}")
print(f"Total Misses: {num_misses}")

Total CER: 2.0
Average CER: 0.01
Total WER: 2.0
Average WER: 0.01
Total Matches: 12
Total Misses: 188


<h2>Denoising</h2>

In [14]:
def denoise (imagePath):
    #read as numpyarray in grayscale
    image = cv2.imread(imagePath, 0)

    #denoising with fastNIMeansDenoising
    dst = cv2.fastNlMeansDenoising(image, None, 30, 7, 21)

    #save image 
    cv2.imwrite('denoised_image.jpg', dst)
    
    #return cv2 as image for pytesseract
    return Image.fromarray(dst)


In [15]:
image = denoise('noise/form page 1-noise-185.jpg')

In [54]:
#image = Image.open('denoised_image.jpg')
#image = 'noise/form page 1-noise-185.jpg'
text = pytesseract.image_to_string(image)
text.split()

['Form',
 'No.',
 'UG-09',
 '|',
 'Office',
 'of',
 'Admissions',
 'and',
 'Scholarships',
 'EXPRESSION',
 'OF',
 'INTEREST',
 'FOR',
 'ADMISSION',
 'Last',
 'Name',
 'i',
 'Rylatt',
 '|',
 '|',
 'First',
 'Name',
 ':',
 '|',
 'a',
 '~',
 'a',
 '%',
 'Brett',
 'oe',
 'ee',
 '|MiadleName',
 '|',
 'o',
 'Fie.',
 '|.',
 'ee',
 'lee',
 'School',
 ':',
 '|',
 '.',
 ':',
 'Far',
 'Eastem',
 'University',
 '—',
 'gore',
 'High',
 'School',
 '~',
 '‘',
 '=',
 '|',
 'To',
 'the',
 'Admissions',
 'Committee:',
 '|',
 'Basa',
 'Seren',
 'red',
 'ae',
 'oe',
 'atari',
 'eu',
 'en',
 'Ae',
 'iaY',
 '=',
 '*',
 '=',
 '-',
 ':',
 '|',
 '|',
 '|',
 'have',
 'been',
 'waitilsted',
 'to',
 'a',
 'degree',
 'program',
 'different',
 'from',
 'my',
 'original',
 'choice/s.',
 '{',
 '|',
 '|',
 'would',
 'like',
 'to',
 'express',
 'my',
 'interest',
 'in',
 'pursuing',
 'my',
 'application',
 'for',
 'admission',
 'under',
 'the',
 'degree',
 '|',
 '|',
 '|',
 '|',
 '|',
 'program',
 'offered',
 'to',
 'm

<h3>Sample Run </h3>

In [19]:
#check if "Last Name" is found. If match, compute error rate; otherwise, just note na hindi kita. Na-affect ng noise
def compute_error(text, csv_idx):
    match = 0
    if "Last Name" in text:
        match = 1
        #split the text into words
        words = text.split()
        
        if "Name" not in words:
            return [0, 0, 0]
        start_idx = words.index("Name")

        #get length of last name from csv
        csv_last_name = last_names[csv_idx]
        csv_last_name_length = len(csv_last_name)
        num_words = len((csv_last_name.split(" ")))
        #extract last name form text
        found_last_name = " ".join(words[start_idx + 1: start_idx + 1 + num_words])
        

        #compute character error rate 
        char_diff = Levenshtein.distance(found_last_name, csv_last_name)
        cer = char_diff/csv_last_name_length
        
        #compute word error rate
        word_diff = Levenshtein.distance(found_last_name.split(), csv_last_name.split())
        word_count = len(csv_last_name.split())
        wer = word_diff/word_count
        return [cer, wer, match]
    return [0, 0, match]

In [20]:
from matplotlib import pyplot as plt
#load image
folder = 'Mail Merged Forms with Effects/'
bright_pattern = 'form page 1-noise-{0:03d}.jpg'

#init var 
total_cer = 0 
total_wer = 0
num_images = 200
num_matches = 0

for i in range (1, num_images+1):
    filename = bright_pattern.format(i)
    image_path = os.path.join(folder, filename)

    if not os.path.exists(image_path):  # Check if file exists
        print(f"Skipping missing file: {filename}")
        continue

    #denoise image 
    image = denoise(image_path)

    #set bounding box
    bbox_form = (0, 370, image.size[0], 1140) # Bounding Box for 02021e
    bbox_image = image.crop(bbox_form)
    

    #run OCR and compute error
    text = pytesseract.image_to_string(bbox_image)
    err = compute_error(text, i-1)
    cer = err[0]
    wer = err[1]
    num_matches += err[2]
    num_misses = 200-num_matches
    
    total_cer += cer
    total_wer += wer
    

print(f"Total CER: {total_cer}")
average_cer = total_cer/num_images
print(f"Average CER: {average_cer}")
print(f"Total WER: {total_wer}")
average_wer = total_wer/num_images
print(f"Average WER: {average_wer}")
print(f"Total Matches: {num_matches}")
print(f"Total Misses: {num_misses}")

Total CER: 33.984920634920634
Average CER: 0.16992460317460317
Total WER: 37.0
Average WER: 0.185
Total Matches: 118
Total Misses: 82
